In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn import preprocessing 
from sklearn.preprocessing import LabelEncoder
%matplotlib inline

from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, f1_score, precision_score

from sklearn import model_selection
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB,BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
import lightgbm as lgb

from sklearn import datasets, linear_model, metrics
from sklearn.model_selection import GridSearchCV
import sklearn.model_selection as ms
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import roc_auc_score

import warnings
warnings.filterwarnings('ignore')

### load data

In [2]:
data = pd.read_csv("./RNA_seq.csv")
dataT=np.array(data)
data=dataT.T
co=data[0]
data1=np.delete(data,0,axis=0)
data=data1
datadf= pd.DataFrame(data=data[0:,0:],columns=co)
datadf.head()
data=datadf
data= data.replace("NOTLC",value=0)
data= data.replace("LC",value=1)
X_yuan=data.drop(['Group'],axis=1)
y_yuan=data['Group']

In [3]:
### LOF
df = data
df = df.dropna()
df = df.drop_duplicates()
from sklearn.neighbors import LocalOutlierFactor
lof = LocalOutlierFactor(n_neighbors=10, contamination=0.1)  # contamination 为异常值比例
lof_predictions = lof.fit_predict(df)
# 将预测结果添加到 DataFrame 中
df['LOF_Predictions'] = lof_predictions
# 筛选出离群值（预测结果为 -1 的点）
outliers = df[df['LOF_Predictions'] == -1]
print("离群值：")
print(outliers)
# 删除离群值所在的行
df_cleaned = df[df['LOF_Predictions'] == 1]  # 保留正常值
df_cleaned = df_cleaned.drop(columns=['LOF_Predictions'])  # 删除辅助列
print("删除离群值后的数据：")
print(df_cleaned)
df = df_cleaned
X=df.drop(['Group'],axis=1)
y=df['Group']

离群值：
      Group ENSG00000000003.15 ENSG00000000005.6 ENSG00000000419.13  \
8         1               7472                 1               4818   
26        1                791                 0                881   
31        1               1339                 0               3032   
37        1               1097                 1               1486   
56        1               2020                 1               1419   
...     ...                ...               ...                ...   
1027      1              10623                 3               4224   
1044      1               9468                 1               2420   
1055      1               4513                 0               4348   
1076      1               8401                 2               4145   
1081      1               2065                 0               5496   

     ENSG00000000457.14 ENSG00000000460.17 ENSG00000000938.13  \
8                   720                931                385   
26          

In [4]:
### standard
column=X.columns
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_pre=sc.fit_transform(X)
X_pre=pd.DataFrame(data=X_pre,columns=column)

In [5]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=None)
print(f'Train shape : {X_train.shape}\nTest shape: {X_test.shape}')

Train shape : (790, 60660)
Test shape: (198, 60660)


In [6]:
def crosspredict(estimator,Xtrain,ytrain,cv):
    print("cross-validate across the entire data set")
    y_pred_cross=cross_val_predict(estimator,Xtrain,ytrain,cv=cv)
    confusion_cross=confusion_matrix(ytrain,y_pred_cross)
    a=accuracy_score(ytrain,y_pred_cross)
    p=precision_score(ytrain, y_pred_cross)
    r=recall_score(ytrain, y_pred_cross)
    f1=f1_score(ytrain, y_pred_cross)
    wf1=f1_score(ytrain, y_pred_cross, average='weighted')
    #auc=roc_auc_score(ytrain,estimator.predict_proba(Xtrain)[:,1])
    print('the confusion_matrix of the model is : \n',confusion_cross)
    print('the accuracy of the model is : ',a)
    print("the precision score of the model is : ", p)
    print("the recall score of the model is :", r)
    print('the f1_score of the model  is :',f1)
    print('the weighted_f1 of the model is :',wf1)
    print('the classification_report is :\n',classification_report(ytrain, y_pred_cross,digits=4))
    #print('the auc is :',auc)
    return a,p,r,f1

### feature selection

In [8]:
from sklearn.feature_selection import SequentialFeatureSelector
def searchNFeatures(estimator,nEnd = 11):
    n_feature_range = range(3,nEnd+1)
    accmax2=0
    premax2=0
    recmax2=0
    f1max2=0
    featuremax2=0
    feature2=[]
    for n in n_feature_range:
        print("N features:", n)
        sel_seq = SequentialFeatureSelector(estimator=estimator, n_features_to_select=n)
        temp=sel_seq.fit(X_train, y_train)
        sel_seq_mask= X_train.columns[temp.get_support()]
        print('features:',sel_seq_mask)
        X_new=sel_seq.fit_transform(X_train,y_train)
        acc2,pre2,rec2,f12=crosspredict(estimator,X_new,y_train,10)
        if f12>f1max2:
            f1max2=f12
            featuremax2=n
            accmax2=acc2
            premax2=pre2
            recmax2=rec2
            feature2=sel_seq_mask
    
    print('feature :',feature2)
    print('the best feature number is:',featuremax2)
    print('the best accuracy is:',accmax2)
    print('the best precision is:',premax2)
    print('the best recall is:',recmax2)
    print('the best f1_score is:',f1max2)
    print("\n")

In [ ]:
searchNFeatures(SVC(kernel="poly",degree=3,coef0=1,probability=True),10)

'ENSG00000223982.3','ENSG00000002726.21','ENSG00000039068.19','ENSG00000106089.12','ENSG00000182685.7','ENSG00000154813.10','ENSG00000204305.14' ,'ENSG00000135604.10', 'ENSG00000004399.13','ENSG00000000003.15'

In [ ]:
searchNFeatures(RandomForestClassifier(max_depth=4,n_estimators=108,n_jobs=-1,random_state=90),10)

'ENSG00000106089.12', 'ENSG00000204305.14', 'ENSG00000223982.3','ENSG00000229693.2'

In [23]:
searchNFeatures(KNeighborsClassifier(n_neighbors=5),10)

'ENSG00000223982.3', 'ENSG00000229693.2', 'ENSG00000091262.16','ENSG00000204305.14', 'ENSG00000004399.13', 'ENSG00000039068.19'

In [ ]:
searchNFeatures(DecisionTreeClassifier(max_depth=8, min_samples_leaf=1, min_samples_split=6),10)

'ENSG00000003147.19','ENSG00000004455.17','ENSG00000006327.14','ENSG00000018510.17','ENSG00000067064.11','ENSG00000154813.10','ENSG00000204305.14','ENSG00000234425.1','ENSG00000262772.2','ENSG00000272477.1'

In [ ]:
searchNFeatures(AdaBoostClassifier(learning_rate=1,n_estimators=50),10)

'ENSG00000000005.6', 'ENSG00000002933.9','ENSG00000019144.19', 'ENSG00000070366.14','ENSG00000091262.16','ENSG00000106089.12','ENSG00000204305.14','ENSG00000233117.4'

In [1]:
searchNFeatures(GaussianNB(),10)

'ENSG00000000938.13', 'ENSG00000004399.13','ENSG00000166123.14', 'ENSG00000204305.14', 'ENSG00000211643.2','ENSG00000259884.1', 'ENSG00000279940.1'

In [ ]:
searchNFeatures(LogisticRegression(C=0.5),10)

'ENSG00000102547.19','ENSG00000135604.10','ENSG00000159352.16','ENSG00000224215.1','ENSG00000234481.2','ENSG00000252275.1','ENSG00000271555.1'

In [ ]:
searchNFeatures(GradientBoostingClassifier(learning_rate=0.01,loss='exponential',n_estimators=300),10)

'ENSG00000000005.6','ENSG00000000971.16', 'ENSG00000001631.16','ENSG00000002587.10', 'ENSG00000004779.10','ENSG00000106089.12', 'ENSG00000144130.11', 'ENSG00000204305.14'

In [ ]:
searchNFeatures(XGBClassifier(booster='gbtree',learning_rate=0.01,n_estimators=400),10)

'ENSG00000008128.23','ENSG00000106089.12','ENSG00000170989.10','ENSG00000006327.14','ENSG00000018510.17','ENSG00000204305.14'

In [ ]:
searchNFeatures(lgb.LGBMClassifier(max_depth=6,n_estimators=200),10)

'ENSG00000106089.12', 'ENSG00000170989.10','ENSG00000204305.14'